In [1]:
from pyspark.sql import SparkSession

# Forzamos cierre de sesiones previas con versiones erróneas
try: spark.stop()
except: pass

spark = SparkSession.builder \
    .appName("Analisis_BigData_Final") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/TiendaBigData.AmazonLaptops") \
    .getOrCreate()

df = spark.read.format("mongodb").load()
print(f"Éxito total: {df.count()} registros.")
df.show(5)

Éxito total: 131 registros.
+--------------------+-------------------+------------------+--------------------+-----+
|                 _id|      fecha_captura|             grupo|       identificador|valor|
+--------------------+-------------------+------------------+--------------------+-----+
|69e408dc780d93a4a...|2026-04-18 22:41:56|Finanzas y mercado|15.6" FHD Laptop ...|399.0|
|69e408dc780d93a4a...|2026-04-18 22:41:56|Finanzas y mercado|Laptop 15.6 Inch ...|349.0|
|69e408dc780d93a4a...|2026-04-18 22:41:56|Finanzas y mercado|Laptop 15.6" Full...|299.0|
|69e408dc780d93a4a...|2026-04-18 22:41:56|Finanzas y mercado|Laptop Ultralight...|369.0|
|69e408dc780d93a4a...|2026-04-18 22:41:56|Finanzas y mercado|Laptop 14 inch Du...|200.0|
+--------------------+-------------------+------------------+--------------------+-----+
only showing top 5 rows



In [2]:
from pyspark.sql.functions import col, split, when, avg
# 2. Transformación de Negocio: Extraer la MARCA
# En Amazon, la primera palabra del título suele ser la marca.
df_transformado = df.withColumn("marca", split(col("identificador"), " ")[0])

# 3. Limpieza de Outliers: Filtrar laptops con precios erróneos (ej: < 100 euros)
df_validado = df_transformado.filter(col("valor") > 100)

# 4. Agregación: Precio promedio por Marca
reporte_marcas = df_validado.groupBy("marca").agg(
    avg("valor").alias("precio_promedio")
).orderBy(col("precio_promedio").desc())

reporte_marcas.show()

+---------+-----------------+
|    marca|  precio_promedio|
+---------+-----------------+
|      MSI|           1558.0|
|       LG|           1474.0|
|     ASUS|844.5714285714286|
|     Acer|            833.4|
|   Lenovo|798.2272727272727|
|Microsoft|            770.0|
|     DELL|            749.0|
|     Dell|            725.0|
| ACEMAGIC|507.2352941176471|
|     acer|            505.2|
|  Lenovo,|            492.5|
|Ordenador|            399.0|
|        2|            399.0|
|      GMR|            393.0|
|    15.6"|            386.5|
|       HP|383.6666666666667|
|   Laptop|382.6111111111111|
|    Ryzen|            379.0|
|     15.6|            351.5|
|  Tunhail|            239.0|
+---------+-----------------+
only showing top 20 rows

